In [1]:
from tabulate import tabulate
import json
import ijson 
import glob
import os
import pandas as pd
import requests
import numpy as np
from datetime import datetime
import re

In [2]:
files = glob.glob('device-event-*.json')
all_events = []

for file in files:
    with open(file, 'rb') as f:
        # Stream each item inside "results"
        for event in ijson.items(f, 'results.item'):
            all_events.append(event)

print("Files found:", len(files))
print("Total records:", len(all_events))

Files found: 10
Total records: 787323


In [3]:
# Build the dataframe

df_raw = pd.DataFrame(all_events)
print(df_raw.shape)

(787323, 90)


In [4]:
df_raw.columns

Index(['manufacturer_contact_zip_ext', 'manufacturer_g1_address_2',
       'event_location', 'report_to_fda', 'manufacturer_contact_t_name',
       'manufacturer_contact_state', 'manufacturer_link_flag',
       'manufacturer_contact_address_2', 'manufacturer_g1_city',
       'manufacturer_contact_address_1', 'manufacturer_contact_pcity',
       'event_type', 'report_number', 'type_of_report', 'product_problem_flag',
       'date_received', 'manufacturer_address_2', 'pma_pmn_number',
       'date_of_event', 'reprocessed_and_reused_flag',
       'manufacturer_address_1', 'exemption_number',
       'manufacturer_contact_zip_code', 'reporter_occupation_code',
       'manufacturer_contact_plocal', 'noe_summarized',
       'manufacturer_contact_l_name', 'source_type',
       'distributor_zip_code_ext', 'manufacturer_g1_postal_code',
       'manufacturer_g1_state', 'reporter_country_code',
       'manufacturer_contact_area_code', 'date_added',
       'manufacturer_contact_f_name', 'device_dat

<div style="text-align: center; font-weight: bold;">Flattening = define the schema</div>

 The flattening logic handle inconsistent JSON structures, like missing fields, strings where dictionaries should be, and empty or malformed lists, so every table (events, devices, patients, narratives) is generated with a stable, predictable schema. This step is important because MAUDE data is messy and inconsistent at scale, and enforcing a consistent structure during flattening ensures the cleaning, modeling, and analysis stages work reliably without errors.

In [5]:
events = df_raw.to_dict("records")

event_rows = []

for e in events:
    if not isinstance(e, dict):
        continue

    event_rows.append({
        "report_number": e.get("report_number"),
        "date_received": e.get("date_received"),
        "date_of_event": e.get("date_of_event"),
        "report_source_code": e.get("report_source_code"),
        "event_type": e.get("event_type"),
        "manufacturer_name": e.get("manufacturer_name"),
        "reporter_state_code": e.get("reporter_state_code"),
        "adverse_event_flag": e.get("adverse_event_flag"),
        "product_problem_flag": e.get("product_problem_flag"),
    })

df_events = pd.DataFrame(event_rows)
len(df_events)

787323

In [6]:
device_rows = []

for e in events:
    report_number = e.get("report_number")
    devices = e.get("device", [])

    # Ensure it's a list
    if not isinstance(devices, list):
        continue

    for d in devices:
        # Skip malformed entries
        if not isinstance(d, dict):
            continue

        device_rows.append({
            "report_number": report_number,
            "device_event_key": d.get("device_event_key"),
            "brand_name": d.get("brand_name"),
            "generic_name": d.get("generic_name"),
            "manufacturer_d_name": d.get("manufacturer_d_name"),
            "device_operator": d.get("device_operator"),
            "device_availability": d.get("device_availability"),
            "device_report_product_code": d.get("device_report_product_code"),

            # This enforce missing columns
            "device_class": d.get("openfda", {}).get("device_class"),
            "medical_specialty": d.get("openfda", {}).get("medical_specialty_description"),
            "regulation_number": d.get("openfda", {}).get("regulation_number")
        })

df_devices = pd.DataFrame(device_rows)


In [7]:
patient_rows = []

for e in events:
    report_number = e.get("report_number")
    patients = e.get("patient", [])

    if not isinstance(patients, list):
        continue

    for p in patients:
        if not isinstance(p, dict):
            continue

        outcomes = p.get("sequence_number_outcome", [])
        if not isinstance(outcomes, list):
            outcomes = []

        patient_rows.append({
            "report_number": report_number,
            "patient_sequence_number": p.get("patient_sequence_number"),
            "patient_age": p.get("patient_age"),
            "patient_sex": p.get("patient_sex"),
            "patient_weight": p.get("patient_weight"),
            "outcome": ",".join(outcomes),
        })

df_patients = pd.DataFrame(patient_rows)

In [8]:
narrative_rows = []

for e in events:
    report_number = e.get("report_number")
    narratives = e.get("mdr_text", [])

    if not isinstance(narratives, list):
        continue

    for n in narratives:
        if not isinstance(n, dict):
            continue

        narrative_rows.append({
            "report_number": report_number,
            "text_type": n.get("text_type_code"),
            "text": n.get("text"),
        })

df_narratives = pd.DataFrame(narrative_rows)


<div style="text-align: center; font-weight: bold;">Cleaning = transform the schema</div>

In [9]:
# -----------------------------
# 1. CLEAN EVENTS TABLE
# -----------------------------

# Define date fields
event_date_cols = [
    "date_received", "date_of_event", "report_date",
    "date_added", "date_changed", "date_facility_aware"
]
# Convert date fields to datetime
for col in event_date_cols:
    if col in df_events.columns:
        df_events[col] = pd.to_datetime(df_events[col], errors="coerce")

# Clean state codes
def clean_state(x):
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    if x in ["", "UNK", "INVALID DATA", "NO DATA", "NONE"]:
        return None
    return x

df_events["reporter_state_code"] = df_events["reporter_state_code"].apply(clean_state)

# Standardize boolean flags
def clean_flag(x):
    if x in ["Y", "YES", "Yes"]:
        return True
    if x in ["N", "NO", "No"]:
        return False
    return None

df_events["adverse_event_flag_clean"] = df_events["adverse_event_flag"].apply(clean_flag)
df_events["product_problem_flag_clean"] = df_events["product_problem_flag"].apply(clean_flag)

# Clean manufacturer name
df_events["manufacturer_name"] = (
    df_events["manufacturer_name"]
    .astype(str)
    .str.upper()
    .str.strip()
    .replace({"NAN": None, "": None})
)

In [10]:
# -----------------------------
# 2. CLEAN DEVICES TABLE
# -----------------------------

# Clean device class
if "device_class" in df_devices.columns:
    df_devices["device_class"] = (
        df_devices["device_class"]
        .astype(str)
        .str.replace("nan", "", regex=False)
        .str.strip()
        .replace({"": None})
    )

# Clean medical specialty
if "medical_specialty" in df_devices.columns:
    df_devices["medical_specialty"] = (
        df_devices["medical_specialty"]
        .astype(str)
        .str.title()
        .replace({"Nan": None})
    )

# Clean manufacturer device name
if "manufacturer_d_name" in df_devices.columns:
    df_devices["manufacturer_d_name"] = (
        df_devices["manufacturer_d_name"]
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({"NAN": None, "": None})
    )

# Clean product code
if "device_report_product_code" in df_devices.columns:
    df_devices["device_report_product_code"] = (
        df_devices["device_report_product_code"]
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({"NAN": None, "": None})
    )

In [11]:
# -----------------------------
# 3. CLEAN PATIENTS TABLE
# -----------------------------

# Extract numeric age
def clean_age(x):
    if pd.isna(x):
        return None
    match = re.search(r"(\d+)", str(x))
    return int(match.group(1)) if match else None

if "patient_age" in df_patients.columns:
    df_patients["patient_age_clean"] = df_patients["patient_age"].apply(clean_age)
else:
    df_patients["patient_age_clean"] = None


# Clean patient sex
if "patient_sex" in df_patients.columns:
    df_patients["patient_sex"] = (
        df_patients["patient_sex"]
        .astype(str)
        .str.upper()
        .replace({"": None, "NAN": None})
    )

# Clean outcome list into a single string
if "outcome" in df_patients.columns:
    df_patients["outcome"] = (
        df_patients["outcome"]
        .astype(str)
        .str.replace("['", "", regex=False)
        .str.replace("']", "", regex=False)
    )
else:
    df_patients["outcome"] = None

In [12]:
# -----------------------------
# 4. CLEAN NARRATIVES TABLE
# -----------------------------

if "text_clean" in df_narratives.columns:
    df_narratives["text_clean"] = (
        df_narratives["text"]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.strip()
    )

In [13]:
# -----------------------------
# 5. STANDARDIZE JOIN KEYS
# -----------------------------

for df in [df_events, df_devices, df_patients, df_narratives]:
    if "report_number" in df.columns:
        df["report_number"] = df["report_number"].astype(str).str.strip()
    else:
        # Ensure the column exists, even if empty
        df["report_number"] = None


QA

In [14]:
# Basic row counts
print("Events:", len(df_events))
print("Devices:", len(df_devices))
print("Patients:", len(df_patients))
print("Narratives:", len(df_narratives))

Events: 787323
Devices: 787322
Patients: 787323
Narratives: 1914136


In [15]:
# Check for missing report number
for name, df in {
    "Events": df_events,
    "Devices": df_devices,
    "Patients": df_patients,
    "Narratives": df_narratives
}.items():
    missing = df["report_number"].isna().sum()
    print(f"{name}: missing report_number = {missing}")


Events: missing report_number = 0
Devices: missing report_number = 0
Patients: missing report_number = 0
Narratives: missing report_number = 0


In [16]:
# Check for duplicates
dupes = df_events["report_number"].duplicated().sum()
print("Duplicate report_number in Events:", dupes)


Duplicate report_number in Events: 34458


In [17]:
df_events.head()

,report_number,date_received,date_of_event,report_source_code,event_type,manufacturer_name,reporter_state_code,adverse_event_flag,product_problem_flag,adverse_event_flag_clean,product_problem_flag_clean
0,3013756811-2025-37708,2025-02-20,2025-02-11,Manufacturer report,Malfunction,None,FL,N,Y,False,True
1,3004753838-2025-070376,2025-03-25,2025-02-27,Manufacturer report,Malfunction,None,MA,N,Y,False,True
2,3005990499-2025-70295,2025-02-24,2024-12-17,Manufacturer report,Injury,None,NH,Y,N,True,False
3,3004753838-2025-041840,2025-02-20,2025-01-06,Manufacturer report,Malfunction,None,None,N,Y,False,True
4,3001617766-2025-006103,2025-03-25,2025-03-06,Manufacturer report,Injury,None,None,Y,N,True,False


In [18]:
# Check for duplicates
dupes = df_events.duplicated(subset=["report_number", "date_of_event"]).sum()
print("Duplicate (report_number, date_of_event) rows:", dupes)


Duplicate (report_number, date_of_event) rows: 5852


In [19]:
# Check for duplicates
dupes = df_events.duplicated(subset=["report_number", "date_received", "date_of_event"]).sum()
print("Duplicate (report_number, date_received, date_of_event) rows:", dupes)

Duplicate (report_number, date_received, date_of_event) rows: 5748


In [20]:
# Validate date fields
df_events["date_received_parsed"] = pd.to_datetime(df_events["date_received"], errors="coerce")
df_events["date_of_event_parsed"] = pd.to_datetime(df_events["date_of_event"], errors="coerce")

print("Invalid date_received:", df_events["date_received_parsed"].isna().sum())
print("Invalid date_of_event:", df_events["date_of_event_parsed"].isna().sum())

Invalid date_received: 0
Invalid date_of_event: 49320


In [21]:
# Check NULL values
critical_cols = ["event_type", "manufacturer_name", "product_problem_flag", "date_received", "date_of_event"]

for col in critical_cols:
    if col in df_events.columns:
        print(col, "nulls:", df_events[col].isna().mean())


event_type nulls: 0.0
manufacturer_name nulls: 1.0
product_problem_flag nulls: 0.0
date_received nulls: 0.0
date_of_event nulls: 0.06264265111015428


In [22]:
# Check device class distribution
if "device_class" in df_devices.columns:
    print(df_devices["device_class"].value_counts(dropna=False).head(20))


device_class
2       568926
3       177952
1        37244
U         1980
f          802
N          411
None         7
Name: count, dtype: int64


Export the files

In [23]:
# Create folder inside your project directory
output_dir = "final_data"
os.makedirs(output_dir, exist_ok=True)

# Build file paths
events_path = os.path.join(output_dir, "events_clean.csv")
devices_path = os.path.join(output_dir, "devices_clean.csv")
patients_path = os.path.join(output_dir, "patients_clean.csv")
narratives_path = os.path.join(output_dir, "narratives_clean.csv")

# Append instead of overwrite
df_events.to_csv(events_path, index=False, mode="a", header=not os.path.exists(events_path))
df_devices.to_csv(devices_path, index=False, mode="a", header=not os.path.exists(devices_path))
df_patients.to_csv(patients_path, index=False, mode="a", header=not os.path.exists(patients_path))
df_narratives.to_csv(narratives_path, index=False, mode="a", header=not os.path.exists(narratives_path))

print("Cleaned tables appended successfully.")

Cleaned tables appended successfully.
